# Notebook 05a · Item2Vec
**Input :**
```
F:\mmd\data\features\item2vec_corpus.pkl
F:\mmd\data\features\feature_config.json
F:\mmd\data\features\gru4rec_val.pkl   ← dùng chung để evaluate
F:\mmd\data\features\gru4rec_test.pkl
```
**Output:**
```
F:\mmd\models\item2vec\
    item2vec.model              ← gensim Word2Vec model
    item_embeddings.npy         ← numpy array shape (N_ITEMS, DIM)
    faiss_index.bin             ← FAISS IVFFlat index cho ANN lookup
    item2vec_results.json       ← Hit@K, NDCG@K trên test set
```
---
### Ý tưởng
```
train_seq = [item_3, item_17, item_42, item_8, ...]
               ↕ treat as "sentence"
Word2Vec skip-gram học embedding sao cho:
  item co-occur trong cùng window → embedding gần nhau

Inference: given basket [i1, i2, i3]
  → mean_vec = mean(emb[i1], emb[i2], emb[i3])
  → top-K nearest neighbors in FAISS index
```

## 0 · Imports & config

In [1]:
# !pip install gensim faiss-cpu
import gc, json, pickle, time, warnings
from pathlib import Path
import numpy as np
import psutil
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

from gensim.models import Word2Vec
import faiss

def ram():
    v = psutil.virtual_memory()
    return f"RAM {v.used/1e9:.1f}/{v.total/1e9:.1f} GB ({v.percent:.0f}%)"

# ── Paths ──────────────────────────────────────────────────────────────
ROOT_DIR    = Path(r"F:\amazon_data")
FEATURE_DIR = ROOT_DIR / "data" / "features"
MODEL_DIR   = ROOT_DIR / "models" / "item2vec"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Load config ────────────────────────────────────────────────────────
with open(FEATURE_DIR / "feature_config.json") as f:
    CFG = json.load(f)

N_ITEMS     = CFG["N_ITEMS"]      # bao gồm PAD (idx=0)
PAD_IDX     = CFG["PAD_IDX"]      # 0
RANDOM_SEED = CFG["RANDOM_SEED"]

# ── Hyperparameters Item2Vec ───────────────────────────────────────────
EMBED_DIM   = CFG["item2vec_dim"]     # 128
WINDOW      = CFG["item2vec_window"]  # 5
MIN_COUNT   = 5    # bỏ item xuất hiện < 5 lần trong corpus
N_WORKERS   = 4    # số CPU threads
N_EPOCHS    = 10
NEGATIVE    = 20   # negative samples per positive
SAMPLE      = 1e-4 # subsampling threshold
TOP_K_LIST  = [5, 10, 20]  # đánh giá Hit@K, NDCG@K

print(f"N_ITEMS   : {N_ITEMS:,}")
print(f"EMBED_DIM : {EMBED_DIM}")
print(f"WINDOW    : {WINDOW}")
print(ram())

N_ITEMS   : 667,278
EMBED_DIM : 128
WINDOW    : 5
RAM 6.0/8.4 GB (71%)


## 1 · Load corpus

In [2]:
with open(FEATURE_DIR / "item2vec_corpus.pkl", "rb") as f:
    corpus = pickle.load(f)

print(f"Corpus size     : {len(corpus):,} sentences")
print(f"Sample sentence : {corpus[0][:8]} ...")
total_tokens = sum(len(s) for s in corpus)
print(f"Total tokens    : {total_tokens:,}")
print(ram())

Corpus size     : 2,712,338 sentences
Sample sentence : ['500522', '151774', '419574', '69738', '216510', '215622', '7305', '295'] ...
Total tokens    : 19,500,343
RAM 7.8/8.4 GB (92%)


## 2 · Train Word2Vec (Item2Vec)

In [3]:
import time
from gensim.models.callbacks import CallbackAny2Vec
from tqdm.auto import tqdm

# Định nghĩa Callback để theo dõi tiến trình và cập nhật tqdm
class TqdmCallback(CallbackAny2Vec):
    def __init__(self, total_epochs):
        self.total_epochs = total_epochs
        self.pbar = None
        
    def on_train_begin(self, model):
        self.pbar = tqdm(total=self.total_epochs, desc="Training Item2Vec", unit="epoch")
        
    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        self.pbar.set_postfix({"loss": f"{loss:.4f}"})
        self.pbar.update(1)
        
    def on_train_end(self, model):
        if self.pbar:
            self.pbar.close()

print("Training Item2Vec...")
t0 = time.time()

model = Word2Vec(
    sentences   = corpus,
    vector_size = EMBED_DIM,
    window      = WINDOW,
    min_count   = MIN_COUNT,
    workers     = N_WORKERS,
    epochs      = N_EPOCHS,
    sg          = 1,         # skip-gram (tốt hơn CBOW cho recommendation)
    negative    = NEGATIVE,
    sample      = SAMPLE,
    seed        = RANDOM_SEED,
    compute_loss= True,
    callbacks   = [TqdmCallback(N_EPOCHS)] # Gọi callback tại đây
)

elapsed = time.time() - t0
print(f"Training done in {elapsed/60:.1f} min")
print(f"Vocabulary size : {len(model.wv):,} items")
print(f"Final loss      : {model.get_latest_training_loss():.4f}")
print(ram())

# Save gensim model
model.save(str(MODEL_DIR / "item2vec.model"))
print(f"Model saved → {MODEL_DIR / 'item2vec.model'}")

Training Item2Vec...


Training Item2Vec:   0%|          | 0/10 [00:00<?, ?epoch/s]

Training done in 39.2 min
Vocabulary size : 548,533 items
Final loss      : 126661784.0000
RAM 6.7/8.4 GB (79%)
Model saved → F:\amazon_data\models\item2vec\item2vec.model


## 3 · Extract embedding matrix
Xây dựng matrix shape `(N_ITEMS, EMBED_DIM)` với row 0 = zero vector (PAD).  
Items không có trong gensim vocab (< min_count) → zero vector.

In [4]:
# Khởi tạo zero matrix
embed_matrix = np.zeros((N_ITEMS, EMBED_DIM), dtype="float32")

n_found = 0
for idx in range(1, N_ITEMS):          # 0 = PAD, bỏ qua
    token = str(idx)                   # corpus dùng str(item_idx)
    if token in model.wv:
        embed_matrix[idx] = model.wv[token]
        n_found += 1

n_zero = N_ITEMS - 1 - n_found        # không tính PAD
print(f"Items with embedding : {n_found:,} / {N_ITEMS-1:,}")
print(f"Items zero (< min_count or unseen): {n_zero:,}")
print(f"Embedding matrix shape : {embed_matrix.shape}")

# L2 normalize → cosine similarity = dot product
norms = np.linalg.norm(embed_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1   # tránh chia 0 (PAD và zero vectors)
embed_matrix_norm = embed_matrix / norms

# Save
np.save(MODEL_DIR / "item_embeddings.npy", embed_matrix)
np.save(MODEL_DIR / "item_embeddings_norm.npy", embed_matrix_norm)
print(f"Saved item_embeddings.npy  |  {embed_matrix.nbytes/1e6:.1f} MB")

Items with embedding : 548,533 / 667,277
Items zero (< min_count or unseen): 118,744
Embedding matrix shape : (667278, 128)
Saved item_embeddings.npy  |  341.6 MB


## 4 · Build FAISS index
Dùng `IndexFlatIP` (inner product trên normalized vector = cosine similarity).  
Với N_ITEMS < 1M → Flat index đủ nhanh (<10ms per query).

In [5]:
print("Building FAISS index...")

# Bỏ PAD row (idx=0) khi build index
# Lưu mapping: faiss_id → item_idx (faiss_id = item_idx - 1)
vectors = embed_matrix_norm[1:].copy()   # shape (N_ITEMS-1, EMBED_DIM)

dim   = vectors.shape[1]
index = faiss.IndexFlatIP(dim)           # Inner Product (= cosine sau khi normalize)
index.add(vectors)

print(f"FAISS index built: {index.ntotal:,} vectors")

# Test query
query = embed_matrix_norm[1:2]           # item idx=1
D, I = index.search(query, k=6)         # top-6 (bao gồm chính nó)
print(f"Test ANN query (item_idx=1):")
print(f"  Neighbor item_idx (faiss_id+1): {I[0]+1}")
print(f"  Cosine similarities           : {D[0].round(4)}")

# Save FAISS index
faiss.write_index(index, str(MODEL_DIR / "faiss_index.bin"))
print(f"FAISS index saved → {MODEL_DIR / 'faiss_index.bin'}")
print(ram())

Building FAISS index...
FAISS index built: 667,277 vectors
Test ANN query (item_idx=1):
  Neighbor item_idx (faiss_id+1): [     1 657723 544657 583504 338980 482058]
  Cosine similarities           : [1.     0.7955 0.795  0.7905 0.7888 0.7855]
FAISS index saved → F:\amazon_data\models\item2vec\faiss_index.bin
RAM 7.6/8.4 GB (90%)


## 5 · Inference function

In [6]:
def recommend_item2vec(basket: list[int],
                       embed_matrix_norm: np.ndarray,
                       faiss_index,
                       top_k: int = 10,
                       exclude_seen: bool = True) -> list[int]:
    """
    basket : list of item_idx (đã shifted, không bao gồm PAD=0)
    Trả về top_k item_idx được recommend.
    """
    # Bỏ PAD và item không có embedding
    valid = [i for i in basket if 1 <= i < len(embed_matrix_norm)
             and np.any(embed_matrix_norm[i] != 0)]
    if not valid:
        return []

    # Mean pooling
    query = embed_matrix_norm[valid].mean(axis=0, keepdims=True).astype("float32")
    norm  = np.linalg.norm(query)
    if norm > 0:
        query /= norm

    # ANN search — lấy nhiều hơn để lọc seen
    fetch_k = top_k + len(basket) + 10
    D, I    = faiss_index.search(query, k=min(fetch_k, faiss_index.ntotal))

    # faiss_id → item_idx (shift +1)
    seen    = set(basket) if exclude_seen else set()
    results = []
    for faiss_id in I[0]:
        item = int(faiss_id) + 1
        if item not in seen and item != PAD_IDX:
            results.append(item)
        if len(results) == top_k:
            break
    return results


# Demo
sample_basket = [100, 200, 300]
recs = recommend_item2vec(sample_basket, embed_matrix_norm, index, top_k=10)
print(f"Sample basket   : {sample_basket}")
print(f"Recommended top-10: {recs}")

Sample basket   : [100, 200, 300]
Recommended top-10: [200986, 543380, 640811, 500713, 614153, 531547, 414795, 531972, 584842, 561210]


## 6 · Evaluate on Val set
Dùng **Hit@K** và **NDCG@K** với negative sampling (100 negatives per sample — chuẩn phổ biến).

In [7]:
def evaluate_item2vec(samples, embed_matrix_norm, faiss_index,
                      top_k_list, n_neg=100, seed=42, desc="Eval"):
    """
    samples  : list of (padded_input, target_item)
    n_neg    : số negative items sample ngẫu nhiên (100-way ranking)
    """
    rng = np.random.default_rng(seed)
    hits   = {k: 0 for k in top_k_list}
    ndcgs  = {k: 0.0 for k in top_k_list}
    total  = 0

    # Tăng mininterval lên 1s để giảm chi phí update thanh tiến trình liên tục
    for inp, target in tqdm(samples, desc=desc, mininterval=1.0):
        # Context: bỏ PAD
        basket = [i for i in inp if i != PAD_IDX]
        if not basket:
            continue

        # Mean embedding
        valid = [i for i in basket if np.any(embed_matrix_norm[i] != 0)]
        if not valid:
            continue
        query = embed_matrix_norm[valid].mean(axis=0).astype("float32")
        norm  = np.linalg.norm(query)
        if norm == 0:
            continue
        query /= norm

        # --- TỐI ƯU HÓA KHÚC NÀY: Negative Sampling Siêu Tốc ---
        ignore_set = set(basket)
        ignore_set.add(target)
        
        negs = []
        # Bốc dư ra 20 items để bù trừ những thằng có thể bị trùng
        sampled = rng.integers(1, N_ITEMS, size=n_neg + 20)
        for s in sampled:
            if s not in ignore_set:
                negs.append(s)
            if len(negs) == n_neg:
                break
                
        candidates = np.append(negs, target)   # target ở cuối
        # --------------------------------------------------------

        # Score = cosine similarity
        cand_embs = embed_matrix_norm[candidates]  # (101, dim)
        scores    = cand_embs @ query              # dot product

        # Rank target
        target_score = scores[-1]
        rank = int((scores > target_score).sum()) + 1   # 1-based

        for k in top_k_list:
            if rank <= k:
                hits[k]  += 1
                ndcgs[k] += 1.0 / np.log2(rank + 1)
        total += 1

    return {
        f"Hit@{k}":  round(hits[k]  / total, 4) for k in top_k_list
    } | {
        f"NDCG@{k}": round(ndcgs[k] / total, 4) for k in top_k_list
    } | {"n_samples": total}


# Load val set
with open(FEATURE_DIR / "gru4rec_val.pkl", "rb") as f:
    val_samples = pickle.load(f)

print(f"Val samples: {len(val_samples):,}")
print("Evaluating Item2Vec on val set...")
val_metrics = evaluate_item2vec(val_samples, embed_matrix_norm, index,
                                 TOP_K_LIST, n_neg=100, desc="Val")
print("\nVal metrics (100-way ranking):")
for k, v in val_metrics.items():
    print(f"  {k:<12}: {v}")

Val samples: 2,712,338
Evaluating Item2Vec on val set...


Val:   0%|          | 0/2712338 [00:00<?, ?it/s]


Val metrics (100-way ranking):
  Hit@5       : 0.0384
  Hit@10      : 0.0683
  Hit@20      : 0.13
  NDCG@5      : 0.0247
  NDCG@10     : 0.0342
  NDCG@20     : 0.0496
  n_samples   : 2712267


## 7 · Evaluate on Test set

In [8]:
with open(FEATURE_DIR / "gru4rec_test.pkl", "rb") as f:
    test_samples = pickle.load(f)

print("Evaluating Item2Vec on test set...")
test_metrics = evaluate_item2vec(test_samples, embed_matrix_norm, index,
                                  TOP_K_LIST, n_neg=100, desc="Test")
print("\nTest metrics (100-way ranking):")
for k, v in test_metrics.items():
    print(f"  {k:<12}: {v}")

# Lưu kết quả
results = {
    "model"       : "Item2Vec",
    "hyperparams" : {"embed_dim": EMBED_DIM, "window": WINDOW,
                     "min_count": MIN_COUNT, "epochs": N_EPOCHS,
                     "negative": NEGATIVE},
    "val_metrics" : val_metrics,
    "test_metrics": test_metrics,
}
with open(MODEL_DIR / "item2vec_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved → {MODEL_DIR / 'item2vec_results.json'}")

Evaluating Item2Vec on test set...


Test:   0%|          | 0/2712338 [00:00<?, ?it/s]


Test metrics (100-way ranking):
  Hit@5       : 0.0321
  Hit@10      : 0.059
  Hit@20      : 0.1155
  NDCG@5      : 0.0203
  NDCG@10     : 0.0289
  NDCG@20     : 0.043
  n_samples   : 2712323

Results saved → F:\amazon_data\models\item2vec\item2vec_results.json


## 8 · Embedding quality check (optional)

In [9]:
import pandas as pd

# Load meta để hiển thị tên item
CLEANED_DIR = ROOT_DIR / "data" / "cleaned"
df_meta = pd.read_parquet(CLEANED_DIR / "meta_clean.parquet",
                           columns=["item_idx", "title", "store"])
df_meta["item_idx_shifted"] = df_meta["item_idx"] + 1
idx2title = df_meta.set_index("item_idx_shifted")["title"].to_dict()

# Tìm top-5 tương đồng cho 3 items mẫu
sample_items = [100, 500, 1000]   # chỉnh idx nếu cần
for item in sample_items:
    if item >= N_ITEMS or np.all(embed_matrix_norm[item] == 0):
        continue
    q = embed_matrix_norm[item:item+1]
    D, I = index.search(q.astype("float32"), k=6)
    similar = [(int(i)+1, round(float(d), 4)) for i, d in zip(I[0][1:], D[0][1:])]
    print(f"\nItem {item}: {idx2title.get(item, 'unknown')[:60]}")
    print("  Similar items:")
    for sim_idx, sim_score in similar:
        title = idx2title.get(sim_idx, "unknown")[:55]
        print(f"    [{sim_score:.4f}] {title}")


Item 100: SAFAVIEH California Shag Collection Accent Rug - 2'3" x 5', 
  Similar items:
    [0.5845] SAFAVIEH Florida Shag Collection Area Rug - 5'3" x 7'6"
    [0.4926] Mohawk Home 8' x 11' Non Slip Rug Pad Gripper 1/2 Thick
    [0.4630] SAFAVIEH Lyndhurst Collection Runner Rug - 2'3" x 8', R
    [0.4460] Machine Washable Ottohome Collection Non-Slip Rubberbac
    [0.4366] SAFAVIEH Paris Shag Collection 9' Square Ivory SG511 Ha

Item 500: ZeroWater 23 Cup Ready-Pour Water Filter Pitcher with Meter 
  Similar items:
    [0.5604] ZeroWater 10-Cup Ready-Pour 5-Stage Water Filter Pitche
    [0.5573] ZeroWater 12-Cup Ready-Pour 5-Stage Water Filter Pitche
    [0.5444] Zerowater Replacement Filters for Pitchers (2 Pack)
    [0.5257] ZeroWater 8-Cup Pitcher
    [0.4983] ZeroWater 4-Pack Replacement Filter Cartridges ZR-004, 

Item 1000: SpotBot Pet handsfree Spot and Stain Cleaner with Deep Reach
  Similar items:
    [0.6006] SEIKO 9 Inch Anniversary Mantel Clock with Glass Dome &
    [0.59

---
## Summary

| Hyperparameter | Value | Ghi chú |
|---|---|---|
| `sg=1` | skip-gram | Tốt hơn CBOW cho sparse data |
| `vector_size=128` | 128 | Cân bằng quality vs memory |
| `window=5` | 5 | Capture co-purchase trong basket |
| `negative=20` | 20 | Cao hơn default (5) vì vocab lớn |
| `min_count=5` | 5 | Khớp với k-core threshold |
| FAISS `IndexFlatIP` | exact | Với N < 1M, exact search đủ nhanh |

**→ Notebook 05b:** Train GRU4Rec  
**→ Notebook 06:** So sánh Item2Vec vs GRU4Rec